In [8]:
import pandas as pd 

df = pd.read_csv("res_model.csv")
df.head()

,Model,Type,Train Init Temp,Train Steps,Train Dimension,Train Demand,Train Scheduler,Train Clustering,Dimension,Step,Scheduler,Clustering,Initial Temp,Initial Cost,Final Cost,Gain
0,/Users/jandretti/Documents/PhD/NSA_CVRP_PPO/wa...,Trained,1,100,20,True,step,False,20,100,cyclic,True,1,10.808784,6.056129,4.752656
1,/Users/jandretti/Documents/PhD/NSA_CVRP_PPO/wa...,Trained,1,100,20,True,step,False,20,100,lam,True,1,10.808784,6.043811,4.764974
2,/Users/jandretti/Documents/PhD/NSA_CVRP_PPO/wa...,Trained,1,100,20,True,step,False,20,100,step,True,1,10.808784,5.915554,4.893230
3,/Users/jandretti/Documents/PhD/NSA_CVRP_PPO/wa...,Trained,1,100,20,True,step,False,20,1000,cyclic,True,1,10.808784,5.568490,5.240295
4,/Users/jandretti/Documents/PhD/NSA_CVRP_PPO/wa...,Trained,1,100,20,True,step,False,20,1000,lam,True,1,10.808784,5.496990,5.311795


In [9]:
# Garder uniquement les colonnes spécifiées
new_df = df[['Model','Train Init Temp', 'Train Steps', 'Train Dimension', 'Train Demand', 'Train Scheduler', 'Train Clustering']].copy()
new_df.loc[:, "Model"] = new_df["Model"].apply(lambda x: x.split("/")[-2])
new_df = new_df.drop_duplicates(subset=["Model"])

# Afficher le nouveau DataFrame
new_df.head()

,Model,Train Init Temp,Train Steps,Train Dimension,Train Demand,Train Scheduler,Train Clustering
0,20250206_141623_eygjgypl,1,100,20,True,step,False
162,20250206_150001_n6l44leu,1000,1000,20,True,step,False
324,20250206_182443_zya1og52,100,100,20,True,step,True
486,20250206_190801_1jrf079s,100,1000,20,True,step,False
648,20250206_223451_9neq6qez,1,100,20,True,step,True


In [10]:
new_df.to_csv("model.csv", index=False)

In [1]:
from model import CVRPActorPairs
from problem import CVRP
import torch


cfg = {
    "DEMANDS": True,
    "NB_CLUSTERS_MAX": 5,
    "N_PROBLEMS": 100,
    "DEVICE": "cpu",
    "EMBEDDING_DIM": 16,
    "C1": 10,
    "C2": 16,
    "OR_TOOLS_TIME": 1,
    "STOP_TEMP": 0.01,
    "INNER_STEPS": 1,
    "METHOD": "ppo",
    "REWARD": "immediate",
    "GAMMA": 0.9,
}

actor = CVRPActorPairs()

cfg["PROBLEM_DIM"] = 20
cfg["MAX_LOAD"] = 30
cfg["CLUSTERING"] = False
problem = CVRP(
    cfg["PROBLEM_DIM"],
    100,
    cfg["MAX_LOAD"],
    device=cfg["DEVICE"],
    params=cfg,
)
params = problem.generate_params(mode="test")
params = {k: v.to(cfg["DEVICE"]) for k, v in params.items()}
problem.set_params(params)
# Find initial solutions
init_x = problem.generate_init_x()
state = problem.to_state(init_x, temp=torch.tensor(1.0), time=torch.tensor(1.0))
action, log_probs, mask = actor.sample(state, problem=problem)
log_probs = actor.evaluate(state,action,mask)
log_probs

tensor(-4.7597, grad_fn=<SelectBackward0>)

In [11]:
import torch
import time


# Tenseur initial
tensor = torch.randint(0, 10, (500, 50))
start = time.time()
# Copie du tenseur pour stocker le résultat
output = tensor.clone()

# Parcours des lignes
for i in range(tensor.shape[0]):
    row = tensor[i]
    mask = row != 0  # Détection des valeurs non nulles
    indices = torch.nonzero(mask).squeeze()  # Indices des valeurs non nulles

    if len(indices) > 0:
        start = indices[0]  # Début du premier groupe
        sum_value = 0

        for j in range(len(indices) - 1):
            sum_value += row[indices[j]].item()
            if indices[j + 1] != indices[j] + 1:  # Fin du groupe détectée
                output[i, start : indices[j] + 1] = sum_value
                start = indices[j + 1]
                sum_value = 0

        # Dernier groupe
        sum_value += row[indices[-1]].item()
        output[i, start : indices[-1] + 1] = sum_value
end = time.time() - start
print(output)
print(end)

tensor([[101, 101, 101,  ...,  13,  13,  13],
        [ 11,  11,   0,  ..., 101, 101, 101],
        [108, 108, 108,  ...,  43,  43,  43],
        ...,
        [  0,   8,   0,  ..., 110, 110, 110],
        [ 45,  45,  45,  ...,  10,  10,  10],
        [ 16,  16,  16,  ...,  34,   0,   8]])
tensor(1.7400e+09)


In [16]:
import torch


def sum_groups(tensor):
    # Detect non-zero values
    mask = tensor != 0
    # indices = torch.arange(tensor.shape[1]).expand_as(tensor)

    # Create a marker for group breaks (when the difference between adjacent indices > 1)
    shift = torch.cat(
        [torch.zeros((tensor.shape[0], 1), dtype=torch.bool), mask[:, :-1]], dim=1
    )
    group_change = mask & ~shift  # Start of new groups

    # Assign a unique group identifier (cumsum creates group labels)
    group_ids = torch.cumsum(group_change, dim=1) * mask

    # Sum by group
    sums = torch.zeros_like(tensor)
    sums.scatter_add_(1, group_ids, tensor)

    # Propagate summed values within their respective groups
    output = sums.gather(1, group_ids)

    return output


# Tenseur initial
tensor = torch.randint(0, 10, (500, 50))

# Application de la transformation
start = time.time()
output = sum_groups(tensor)
end = time.time() - start
print(end)

print(output)
print(tensor[0])

0.00045990943908691406
tensor([[ 60,  60,  60,  ...,   0,   8,   8],
        [  3,   3,   0,  ..., 112, 112, 112],
        [ 18,  18,  18,  ..., 119, 119, 119],
        ...,
        [ 41,  41,  41,  ...,  17,  17,   0],
        [ 58,  58,  58,  ...,  37,  37,  37],
        [ 12,  12,   0,  ...,  21,  21,  21]])
tensor([6, 9, 2, 1, 5, 3, 2, 9, 2, 4, 4, 6, 4, 3, 0, 3, 0, 9, 3, 5, 4, 0, 1, 1,
        4, 4, 2, 2, 4, 1, 1, 5, 3, 7, 5, 0, 3, 5, 1, 4, 3, 8, 4, 8, 1, 7, 1, 0,
        1, 7])
